# 04 Retrospective Modeling
Uses post-arrival features for retrospective benchmarking.

Prerequisite: run 01_data_prep.ipynb first.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data_dir = Path('../data')
model_B_df = pd.read_csv(data_dir / 'model_B_retrospective_dataset.csv')

for c in ['WAITTIME', 'YEAR']:
    if c in model_B_df.columns:
        model_B_df[c] = pd.to_numeric(model_B_df[c], errors='coerce')

model_B_df = model_B_df[model_B_df['WAITTIME'].between(0, 480)].copy()
print('model_B_df shape:', model_B_df.shape)

model_B_df shape: (91812, 33)


In [2]:
split_years = {'train': [2015, 2016, 2017, 2018], 'valid': [2021], 'holdout': [2022]}
train_df = model_B_df[model_B_df['YEAR'].isin(split_years['train'])].copy()
valid_df = model_B_df[model_B_df['YEAR'].isin(split_years['valid'])].copy()
holdout_df = model_B_df[model_B_df['YEAR'].isin(split_years['holdout'])].copy()

print('train:', len(train_df), 'valid:', len(valid_df), 'holdout:', len(holdout_df))

train_hour_median = train_df.groupby('ARRIVAL_HOUR', dropna=False)['WAITTIME'].median() if 'ARRIVAL_HOUR' in train_df.columns else pd.Series(dtype=float)
train_global_median = train_df['WAITTIME'].median()

def baseline_predict(frame):
    if 'ARRIVAL_HOUR' not in frame.columns or train_hour_median.empty:
        return pd.Series(np.repeat(train_global_median, len(frame)), index=frame.index)
    return frame['ARRIVAL_HOUR'].map(train_hour_median).fillna(train_global_median)

base_valid = baseline_predict(valid_df)
base_holdout = baseline_predict(holdout_df)

train: 64766 valid: 13830 holdout: 13216


In [3]:
features = [c for c in model_B_df.columns if c not in ['WAITTIME', 'YEAR', 'ARRTIME']]
X_train = train_df[features].copy()
X_valid = valid_df[features].copy()
X_holdout = holdout_df[features].copy()
y_train = train_df['WAITTIME']
y_valid = valid_df['WAITTIME']
y_holdout = holdout_df['WAITTIME']

cat_cols = []
num_cols = []
for c in features:
    if str(model_B_df[c].dtype) in ['object', 'category']:
        cat_cols.append(c)
        X_train[c] = X_train[c].astype('string').fillna('missing')
        X_valid[c] = X_valid[c].astype('string').fillna('missing')
        X_holdout[c] = X_holdout[c].astype('string').fillna('missing')
    else:
        num_cols.append(c)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ],
    remainder='drop'
)

models = {
    'linear_regression': Pipeline([('prep', preprocessor), ('model', LinearRegression())]),
    'random_forest': Pipeline([('prep', preprocessor), ('model', RandomForestRegressor(n_estimators=150, max_depth=18, min_samples_leaf=2, n_jobs=-1, random_state=42))])
}

rows = []
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    p_valid = model.predict(X_valid)
    rows.append({'model_name': name, 'split': 'valid', 'mae': mean_absolute_error(y_valid, p_valid), 'rmse': np.sqrt(mean_squared_error(y_valid, p_valid)), 'r2': r2_score(y_valid, p_valid)})
    fitted[name] = model

res = pd.DataFrame(rows).sort_values(['mae', 'rmse'])
display(res)
best_name = res.iloc[0]['model_name']
best = fitted[best_name]

,model_name,split,mae,rmse,r2
0,linear_regression,valid,32.151865,48.904341,-0.009119
1,random_forest,valid,32.809962,48.048138,0.025906


In [4]:
pred_holdout = best.predict(X_holdout)
eval_df = pd.DataFrame([
    {'model_name': 'by_hour_median_trainfit', 'split': 'holdout', 'mae': mean_absolute_error(y_holdout, base_holdout), 'rmse': np.sqrt(mean_squared_error(y_holdout, base_holdout)), 'r2': r2_score(y_holdout, base_holdout)},
    {'model_name': best_name, 'split': 'holdout', 'mae': mean_absolute_error(y_holdout, pred_holdout), 'rmse': np.sqrt(mean_squared_error(y_holdout, pred_holdout)), 'r2': r2_score(y_holdout, pred_holdout)}
])
display(eval_df)

eval_df.to_csv(data_dir / 'model_eval_summary_retrospective.csv', index=False)
holdout_out = holdout_df.copy()
holdout_out['y_true'] = y_holdout
holdout_out['pred_base'] = base_holdout
holdout_out['pred_best'] = pred_holdout
holdout_out.to_csv(data_dir / 'holdout_predictions_retrospective.csv', index=False)
print('Saved retrospective artifacts to data/')

,model_name,split,mae,rmse,r2
0,by_hour_median_trainfit,holdout,28.344885,55.999156,-0.100491
1,linear_regression,holdout,34.486214,53.475413,-0.003533


Saved retrospective artifacts to data/
